# wav2vec2 — Speech Emotion Recognition (Kaggle)

Replaces the hand-built MFCC+prosody features with a **pretrained speech encoder**
(`facebook/wav2vec2-base`), fine-tuned end-to-end for the 6-class problem. Motivation: the
MFCC pipeline (single-stage ~0.63 and the two-stage variant ~0.62) is capped by **feature-level
overlap** between Fearful / Sad / Neutral — no classifier topology fixed it. wav2vec2 learns
representations that capture emotional prosody far better than MFCCs, which is the standard lever
for pushing CREMA-D / RAVDESS past the MFCC ceiling.

**Kept identical to the MFCC notebooks for a fair comparison:**
- same CREMA-D + RAVDESS subset (TESS / SAVEE and the misspelled `Suprised` dropped),
- same 1–5 s duration filter,
- same **speaker-independent** `GroupShuffleSplit(test_size=0.15, random_state=42)` — so the test
  set is the same clips, and the accuracy is directly comparable to ~0.63.

**Changed:** audio resampled to 16 kHz (wav2vec2's rate), conv feature-encoder frozen, transformer
+ classifier head fine-tuned with class-balanced loss. wav2vec2's built-in SpecAugment masking
provides augmentation during training, so the heavy 3× audio augmentation is dropped.

In [ ]:
# === Setup / imports ===
import sys, subprocess, os
import torch
from pathlib import Path
from packaging.version import parse

# --- Environment detection ---
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB  = "google.colab" in sys.modules
if not ON_COLAB:
    try:
        import google.colab
        ON_COLAB = True
    except ImportError:
        pass

if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working")
    ENV     = "Kaggle"
elif ON_COLAB:
    OUT_DIR = Path("/content")
    ENV     = "Colab"
else:
    OUT_DIR = Path(".")
    ENV     = "Local"

print(f"Environment : {ENV}")
print(f"Output dir  : {OUT_DIR}")

# --- Optional: mount Google Drive on Colab for persistent model saving ---
# Uncomment if you want to save the model to Drive instead of /content (lost on disconnect)
# if ON_COLAB:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     OUT_DIR = Path("/content/drive/MyDrive/w2v2_ser")
#     OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- GPU info ---
print("\ntorch       :", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
print("GPUs        :", N_GPU,
      [torch.cuda.get_device_name(i) for i in range(N_GPU)])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Package check ---
need = False
try:
    import transformers, accelerate
    if parse(transformers.__version__) < parse("4.41") or parse(accelerate.__version__) < parse("0.30"):
        need = True
except ImportError:
    need = True

if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.41", "accelerate>=0.30",
                    f"torch=={torch.__version__}"], check=True)
    print(">> installed/upgraded — RESTART THE RUNTIME and Run All again")
else:
    import transformers
    print("transformers:", transformers.__version__, "-- ok")

import glob, re
import numpy as np
import pandas as pd

In [ ]:
# === Locate dataset + build the clip table (identical filtering to the MFCC notebooks) ===
# NOTE: a committed / "Save Version" run is NON-interactive, and kagglehub.dataset_download() can
# only ATTACH a new dataset in interactive sessions. So we first look for an already-attached copy
# under /kaggle/input (Add Input -> uldisvalainis/audio-emotions), and only fall back to kagglehub
# when running interactively. This is what makes Save & Run All work.
KNOWN_EMOTIONS = {"Angry", "Disgusted", "Fearful", "Happy", "Neutral", "Sad"}

def find_emotions_dir(root):
    for c in glob.glob(os.path.join(str(root), "**", ""), recursive=True):
        try:
            subs = {d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))}
        except OSError:
            continue
        if len(KNOWN_EMOTIONS & subs) >= 3:
            return Path(c)
    return None

# 1) already-attached input dataset (works in committed runs)
EMO_DIR = find_emotions_dir("/kaggle/input") if Path("/kaggle/input").exists() else None
# 2) fall back to kagglehub download (interactive sessions only)
if EMO_DIR is None:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("uldisvalainis/audio-emotions")
    EMO_DIR = find_emotions_dir(DATA_ROOT)
if EMO_DIR is None:
    raise FileNotFoundError(
        "Couldn't find <emotion>/*.wav folders. Attach the dataset via "
        "Add Input -> uldisvalainis/audio-emotions (or any dataset containing the Emotions folders).")
print("data dir:", EMO_DIR)

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df = pd.DataFrame(rows)
df = df[df["source"].isin(["CREMA-D", "RAVDESS"])]
df = df[df["emotion"] != "Suprised"].reset_index(drop=True)
print(f"{len(df)} clips, {df['emotion'].nunique()} emotions")
df["emotion"].value_counts()

In [ ]:
# === Drop clips outside 1-5s (header read, no full decode) ===
import librosa
from tqdm.auto import tqdm

df["duration"] = [librosa.get_duration(path=p) for p in tqdm(df["path"], desc="duration")]
before = len(df)
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 5.0)].reset_index(drop=True)
print(f"removed {before - len(df)} clips  ({before} -> {len(df)} remain)")
df["emotion"].value_counts()

In [ ]:
# === Speaker-INDEPENDENT splits: train / val / test, all speaker-disjoint ===
from sklearn.model_selection import GroupShuffleSplit

def speaker_of(filename, source):
    if source == "CREMA-D":   return f"CREMA-{filename[:4]}"
    if source == "RAVDESS":   return f"RAVDESS-{filename.split('.')[0].split('-')[-1]}"
    return f"{source}-{filename}"

df["speaker"] = [speaker_of(n, s) for n, s in zip(df["filename"], df["source"])]

# 1) hold out TEST -- SAME seed/params as the MFCC notebooks, so the test clips are IDENTICAL
#    (keeps the headline accuracy directly comparable to the ~0.63 baseline and the 0.726 run).
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trv_idx, te_idx = next(gss_test.split(df, df["emotion"], groups=df["speaker"]))
dftrv  = df.iloc[trv_idx].reset_index(drop=True)     # train+val pool
dftest = df.iloc[te_idx].reset_index(drop=True)

# 2) carve a speaker-disjoint VALIDATION set out of the train pool, for checkpoint selection only
#    -- this is what lets us stop peeking at the test set during training.
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, va_idx = next(gss_val.split(dftrv, dftrv["emotion"], groups=dftrv["speaker"]))
dftrain = dftrv.iloc[tr_idx].reset_index(drop=True)
dfval   = dftrv.iloc[va_idx].reset_index(drop=True)

for a, b, nm in [(dftrain, dfval, "train/val"), (dftrain, dftest, "train/test"), (dfval, dftest, "val/test")]:
    assert not (set(a["speaker"]) & set(b["speaker"])), f"speaker leakage in {nm}"
print(f"train {len(dftrain)} / val {len(dfval)} / test {len(dftest)} clips  (all speaker-disjoint)")
print(f"speakers -- train {dftrain['speaker'].nunique()}, "
      f"val {dfval['speaker'].nunique()}, test {dftest['speaker'].nunique()}")

In [ ]:
# === Labels ===
from sklearn.preprocessing import LabelEncoder
le   = LabelEncoder()
ytr  = le.fit_transform(dftrain["emotion"])
yval = le.transform(dfval["emotion"])
yte  = le.transform(dftest["emotion"])
CLASSES = list(le.classes_)
NUM = len(CLASSES)
print("classes:", CLASSES)

In [ ]:
# === Audio pipeline: resample to model rate (16 kHz), preload clips into RAM ===
# WavLM-large (300M params) vs wavlm-base-plus (94M): same API, ~5-8% better on SER benchmarks
# because its 24 transformer layers and wider hidden size (1024 vs 768) capture richer
# emotional prosody. Gradient checkpointing is re-enabled to fit on A100 (40GB).
from transformers import AutoFeatureExtractor

MODEL_ID = "microsoft/wavlm-large"   # was wavlm-base-plus
fe  = AutoFeatureExtractor.from_pretrained(MODEL_ID)
SR      = fe.sampling_rate          # 16000
MAX_LEN = int(SR * 5.0)
print(f"Model  : {MODEL_ID}")
print(f"SR     : {SR}  |  MAX_LEN: {MAX_LEN} samples ({MAX_LEN/SR:.0f}s)")

class WavDataset(torch.utils.data.Dataset):
    def __init__(self, frame, labels, desc, augment=False):
        self.wavs = [librosa.load(p, sr=SR)[0][:MAX_LEN].astype("float32")
                     for p in tqdm(list(frame["path"]), desc=desc)]
        self.labels  = list(labels)
        self.augment = augment
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        wav = self.wavs[i].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                wav += np.random.randn(len(wav)).astype("float32") * 0.003
            if np.random.rand() < 0.3:
                rate = np.random.uniform(0.9, 1.1)
                wav  = librosa.effects.time_stretch(wav, rate=rate)[:MAX_LEN]
                wav  = wav.astype("float32")
        return {"input_values": wav, "label": int(self.labels[i])}

train_ds = WavDataset(dftrain, ytr,  "load-train", augment=True)
val_ds   = WavDataset(dfval,   yval, "load-val")
test_ds  = WavDataset(dftest,  yte,  "load-test")
print("train/val/test items:", len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# === Collator: pad each batch with the feature extractor (also does wav2vec2/WavLM normalization) ===
from dataclasses import dataclass
from transformers import AutoFeatureExtractor

@dataclass
class Collator:
    fe: AutoFeatureExtractor
    def __call__(self, batch):
        wavs   = [b["input_values"] for b in batch]
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        out = self.fe(wavs, sampling_rate=self.fe.sampling_rate, padding=True, return_tensors="pt")
        out["labels"] = labels
        return out

collator = Collator(fe)

In [ ]:
# === Model: WavLM-large + classification head ===
from transformers import WavLMForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

model = WavLMForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM,
    label2id={c: i for i, c in enumerate(CLASSES)},
    id2label={i: c for i, c in enumerate(CLASSES)},
    use_weighted_layer_sum=True,
)
model.freeze_feature_encoder()
# gradient_checkpointing NOT used — A100 (40GB) has enough VRAM for batch=16 without it,
# and skipping recomputation saves ~30% of training time.

cw      = compute_class_weight("balanced", classes=np.arange(NUM), y=ytr)
class_w = torch.tensor(cw, dtype=torch.float32)
print("class weights:", {CLASSES[i]: round(float(w), 3) for i, w in enumerate(cw)})
n_layers  = model.wavlm.config.num_hidden_layers
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone layers : {n_layers}")
print(f"Trainable params: {trainable:,}")

In [ ]:
# === Trainer: LLRD optimizer + focal loss + cosine schedule ===
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score

EFFECTIVE_BATCH  = 32
PER_DEVICE_BATCH = 16    # A100 40GB: batch=16 uses ~5GB total — safe, leaves 35GB headroom
GRAD_ACCUM = max(1, EFFECTIVE_BATCH // (PER_DEVICE_BATCH * max(1, N_GPU)))
print(f"GPUs: {N_GPU} | per_device: {PER_DEVICE_BATCH} | grad_accum: {GRAD_ACCUM} | effective batch: {PER_DEVICE_BATCH * max(1, N_GPU) * GRAD_ACCUM}")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'fp32'}")

# --- LLRD optimizer ---
def get_llrd_optimizer(model, base_lr=3e-5, decay=0.95, wd=0.01):
    no_decay = {"bias", "LayerNorm.weight"}
    params   = []
    head_params = [(n, p) for n, p in model.named_parameters()
                   if "wavlm" not in n and p.requires_grad]
    params.append({"params": [p for _, p in head_params],
                   "lr": base_lr, "weight_decay": wd})
    for i, layer in enumerate(reversed(model.wavlm.encoder.layers)):
        lr_i = base_lr * (decay ** i)
        nd   = [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)]
        wd_p = [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)]
        params += [{"params": wd_p, "lr": lr_i, "weight_decay": wd},
                   {"params": nd,   "lr": lr_i, "weight_decay": 0.0}]
    return torch.optim.AdamW(params)

# --- Focal loss trainer ---
class WeightedTrainer(Trainer):
    def create_optimizer(self):
        self.optimizer = get_llrd_optimizer(self.model, base_lr=3e-5, decay=0.95)
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out    = model(**inputs)
        ce     = torch.nn.functional.cross_entropy(
            out.logits, labels,
            weight=class_w.to(out.logits.device),
            reduction="none",
        )
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** 2 * ce).mean()
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds  = logits.argmax(-1)
    return {"acc":      accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir=str(OUT_DIR / "w2v2_ser"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=3e-5,
    num_train_epochs=20,
    warmup_steps=400,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=use_fp16,
    bf16=use_bf16,
    dataloader_num_workers=4,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    report_to="none",
)

torch.cuda.empty_cache()

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
trainer.train()

In [ ]:
# === Full 6-class evaluation on the speaker-independent TEST set ===
import gc
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

# Free optimizer states + flush fragmented memory
if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
    del trainer.optimizer
    trainer.optimizer = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free after cleanup: {(torch.cuda.mem_get_info()[0]/1024**3):.1f} GB")

# Manual inference — bypass trainer.predict() which accumulates all logits on GPU.
# Each batch is processed and immediately moved to CPU, keeping GPU memory flat.
def run_inference(dataset, batch_size=4):
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator,
                        num_workers=2, pin_memory=False)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="inference"):
            labels = batch.pop("labels", None)
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**inputs).logits.cpu().float()  # offload to CPU immediately
            all_logits.append(logits)
            del inputs, logits
    return torch.cat(all_logits, dim=0).numpy()

test_logits = run_inference(test_ds)
pred = test_logits.argmax(-1)

val_logits  = run_inference(val_ds)
val_pred    = val_logits.argmax(-1)
print(f"VAL  -- acc {accuracy_score(yval, val_pred):.4f}  f1_macro {f1_score(yval, val_pred, average='macro'):.4f}")
print(f"\n--- Test set (speaker-independent) ---")
print(f"Test acc    : {accuracy_score(yte, pred):.4f}")
print(f"F1 macro    : {f1_score(yte, pred, average='macro'):.4f}")
print()
print(classification_report(yte, pred, target_names=CLASSES, digits=3))

print("--- Per-corpus accuracy ---")
for src in ["CREMA-D", "RAVDESS"]:
    mask = dftest["source"].values == src
    acc  = (pred[mask] == yte[mask]).mean()
    print(f"  {src}: {acc:.4f}  ({mask.sum()} clips)")

cm = confusion_matrix(yte, pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("WavLM-large + weighted-layer-sum | speaker-independent test")
plt.show()

In [ ]:
# === Save the fine-tuned model + feature extractor ===
save_path = OUT_DIR / "w2v2_ser_best"
save_path.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(save_path))
fe.save_pretrained(str(save_path))
print(f"Saved to  : {save_path}")
print(f"Model     : {MODEL_ID}")
print(f"Classes   : {CLASSES}")

# On Colab: /content is wiped on disconnect — download the folder or mount Drive above
if ON_COLAB:
    print("\nColab tip: run this to download the model folder:")
    print("  from google.colab import files")
    print("  import shutil")
    print(f"  shutil.make_archive('/content/w2v2_ser_best', 'zip', '{save_path}')")
    print("  files.download('/content/w2v2_ser_best.zip')")